In [21]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_extract, when, lit, udf, regexp_replace, lower, trim, translate, initcap, split
from pyspark.sql.types import FloatType, StringType, IntegerType
import re

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")

spark = (
    SparkSession.builder
    .appName("AutoTec_Batch_Processing")
    .config("spark.mongodb.read.connection.uri", MONGO_URI)
    .config("spark.mongodb.write.connection.uri", MONGO_URI)
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1")
    .getOrCreate()
)

df_raw = (
    spark.read.format("mongodb")
    .option("database", "proyecto_bigdata")
    .option("collection", "bd_autos")
    .load()
)

In [17]:
print(df_raw.count())

3554


### Marca


In [22]:

marca = lower(trim(col("marca")))
# quitar tildes
marca = translate(
    marca,
    "áéíóúüñ",
    "aeiouun"
)
# extraer solo primera palabra
marca = split(marca, " ").getItem(0)

df_marca = df_raw.withColumn("marca_limpia", marca)
df_marca = df_marca.withColumn(
    "marca_limpia",
    when(col("marca_limpia").isin("chevy"), "chevrolet")
    .when(col("marca_limpia").isin("mercedes-benz"), "mercedes")
    .when(col("marca_limpia").isin("vw"), "volkswagen")
    .when(col("marca_limpia") == "citroën", "citroen")
    .otherwise(col("marca_limpia"))
)
marcas_invalidas = [
    "null", "nan","poco","semi","usados","vendo","unico","recien","bajo","cotiza","descuenta"
]
df_marca = df_marca.filter(
    (~col("marca_limpia").isin(marcas_invalidas)) &
    col("marca_limpia").isNotNull()
)
print("Registros después de limpieza:", df_marca.count())

Registros después de limpieza: 3446


### Modelo


In [23]:
df_modelo = df_marca.filter(col("modelo").isNotNull())
df_modelo = df_modelo.withColumn(
    "modelo_limpio",
    trim(col("modelo"))
)
df_modelo = df_modelo.filter(col("modelo_limpio") != "")
df_modelo = df_modelo.withColumn(
    "modelo_limpio",
    regexp_replace(col("modelo_limpio"), r"\s+", " ")
)
df_modelo = df_modelo.withColumn(
    "modelo_limpio",
    initcap(lower(col("modelo_limpio")))
)
print("Registros después de limpieza:", df_modelo.count())

Registros después de limpieza: 3435


### Year


In [24]:
df_year = df_modelo
# Eliminar duplicados por URL
df_year = df_year.dropDuplicates(["url"])
df_year = df_year.filter(col("year").isNotNull())
# Limpiar year y convertir a entero
df_year = df_year.withColumn(
    "year_limpio",
    regexp_replace(col("year"), "[^0-9]", "").cast("int")
)
# Eliminar outliers
df_year = df_year.filter(
    (col("year_limpio") >= 1990) &
    (col("year_limpio") <= 2026)
)
print("Cantidad final limpia:", df_year.count())

Cantidad final limpia: 3434


### Kilometraje


In [25]:
df_km = df_year.filter(
    (col("kilometraje").isNotNull()) &
    (col("kilometraje") != "") &
    (col("kilometraje") != "Sin kilometraje") &
    (col("kilometraje") != "Sin información")
)
df_km = df_km.withColumn(
    "kilometraje_limpio",
    regexp_replace(col("kilometraje"), r"[^\d]", "").cast(FloatType())
)
# eliminacion outliers
df_km = df_km.filter(
    (col("kilometraje_limpio") >= 1000) & (col("kilometraje_limpio") <= 300000)
)
print("Cantidad final limpia:", df_km.count())

Cantidad final limpia: 3328


### Combustible


In [26]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

comb_norm = lower(
    translate(col("combustible"), "áéíóúüñÁÉÍÓÚÜÑ", "aeiouunAEIOUUN")
)

df_combustible = df_km.withColumn(
    "combustible_limpio",
    when(comb_norm.isin("bencina", "gasolina"), "bencina")
    .when(comb_norm.isin("diesel", "diessel"), "diesel")
    .when(comb_norm.isin("hibrido"), "hibrido")
    .when(comb_norm.isin("electrico", "electricidad"), "electrico")
    .when(comb_norm.isin("gnc", "gas-bencina", "gas bencina"), "gnc")
    .otherwise(None)
)

df_combustible = df_combustible.filter(
    col("combustible_limpio").isNotNull() &
    (trim(col("combustible_limpio")) != "")
)

indexer_combustible = StringIndexer(
    inputCol="combustible_limpio",
    outputCol="cat_combustible",
    handleInvalid="keep"
)

pipeline = Pipeline(stages=[indexer_combustible])
df_combustible = pipeline.fit(df_combustible).transform(df_combustible)

df_combustible = df_combustible.withColumn(
    "cat_combustible",
    col("cat_combustible").cast("int")
)

print("Cantidad limpia:", df_combustible.count())

Cantidad limpia: 2612


### Ciudad


In [27]:
df_ciudad = df_combustible.withColumn(
    "ciudad_limpia",
    lower(trim(col("ciudad")))
)
df_ciudad = df_ciudad.withColumn(
    "ciudad_limpia",
    regexp_replace("ciudad_limpia", "á", "a")
)
df_ciudad = df_ciudad.withColumn(
    "ciudad_limpia",
    regexp_replace("ciudad_limpia", "é", "e")
)
df_ciudad = df_ciudad.withColumn(
    "ciudad_limpia",
    regexp_replace("ciudad_limpia", "í", "i")
)
df_ciudad = df_ciudad.withColumn(
    "ciudad_limpia",
    regexp_replace("ciudad_limpia", "ó", "o")
)
df_ciudad = df_ciudad.withColumn(
    "ciudad_limpia",
    regexp_replace("ciudad_limpia", "ú", "u")
)
df_ciudad = df_ciudad.withColumn(
    "ciudad_limpia",
    when(col("ciudad_limpia") == "no disponible", None)
    .when(col("ciudad_limpia") == "otra comuna", None)
    .when(col("ciudad_limpia") == "chile", None)
    .when(col("ciudad_limpia") == "n/a", None)
    .when(col("ciudad_limpia") == "chile (sucursal difor)", "santiago")
    .when(col("ciudad_limpia") == "metropolitana de santiago", "santiago")
    .when(col("ciudad_limpia") == "region metropolitana", "santiago")
    .when(col("ciudad_limpia") == "cerrillos, santiago", "cerrillos")
    .when(col("ciudad_limpia") == "vina del mar", "viña del mar")
    .when(col("ciudad_limpia") == "concepcion", "concepción")
    .otherwise(col("ciudad_limpia"))
)
df_ciudad = df_ciudad.filter(col("ciudad_limpia").isNotNull())
print("Cantidad limpia:", df_ciudad.count())

Cantidad limpia: 1991


### Precio


In [28]:
#  eliminar nulos
df_precio = df_ciudad.na.drop(subset=["precio"])
# normalizar formato y convertir a número 
df_precio = df_precio.withColumn("precio_limpio",
    regexp_replace(col("precio"), "[^0-9]", "").cast("float"))
# eliminar outliers (ejemplo: rango válido entre 500.000 y 200.000.000) 
df_precio = df_precio .filter((col("precio_limpio") >= 500000) & (col("precio_limpio") <= 200000000))

print("Cantidad limpia:", df_precio.count())

Cantidad limpia: 1988


### DF LIMPIO FINAL


In [29]:
print("Cantidad limpia TOTAL:", df_precio.count())
#ELIMINACION DE COLUMNAS SUCIAS
df_precio = df_precio.drop(
    "marca",
    "combustible",
    "ciudad",
    "precio",
    "kilometraje",
    "modelo",
    "year",
    "fuente"
)
df_limpio = df_precio \
    .withColumnRenamed("marca_limpia", "marca") \
    .withColumnRenamed("modelo_limpio", "modelo") \
    .withColumnRenamed("year_limpio", "year") \
    .withColumnRenamed("kilometraje_limpio", "kilometraje") \
    .withColumnRenamed("combustible_limpio", "combustible") \
    .withColumnRenamed("ciudad_limpia", "ciudad") \
    .withColumnRenamed("precio_limpio", "precio")
df_limpio.printSchema()
df_limpio.show(5, truncate=False)


Cantidad limpia TOTAL: 1988
root
 |-- _id: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- foto_url: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- url: string (nullable = true)
 |-- usuario: string (nullable = true)
 |-- marca: string (nullable = true)
 |-- modelo: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- kilometraje: float (nullable = true)
 |-- combustible: string (nullable = true)
 |-- cat_combustible: integer (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- precio: float (nullable = true)

+------------------------+-------------------+--------+-------+------------------------------------------------------------------------------------------------+----------+-----+---------------------------+----+-----------+-----------+---------------+--------+-------+
|_id                     |fecha_captura      |foto_url|grupo  |url                                                                                 

In [30]:
from pyspark.sql import SparkSession
import os
from dotenv import load_dotenv

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")

spark = (
    SparkSession.builder
    .appName("AutoTec_Batch_Processing")
    .config("spark.mongodb.read.connection.uri", MONGO_URI)
    .config("spark.mongodb.write.connection.uri", MONGO_URI)
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1")
    .getOrCreate()
)

df_limpio.write \
    .format("mongodb") \
    .mode("overwrite") \
    .option("database", "proyecto_bigdata") \
    .option("collection", "Contenedor_Autos_Limpio") \
    .save()

print("Datos limpios cargados correctamente")

Datos limpios cargados correctamente
